# Łączenie agenta AI ze światem: wyjście poza cyfrową klatkę

## Cel: Zrozumienie roli narzędzi w architekturze agenta, demistyfikacja standardu MCP (Model Context Protocol) oraz poznanie strategii zarządzania kontekstem przy dużej liczbie akcji


> "The world is full of obvious things which nobody by any chance ever observes"
> — Sherlock Holmes
>
> "Świat jest pełen oczywistych rzeczy, których nikt nigdy nie zauważa"


### Czego się nauczysz:

1. Zrozumiesz fundamentalną różnicę między Hostem, Klientem a Serwerem w architekturze MCP i dowiesz się, dlaczego to Twoja aplikacja jest Hostem
2. Poznasz ewolucję warstwy transportowej i przyczyny, dla których branża porzuciła `Server-Sent Events` na rzecz `Streamable HTTP`
3. Nauczysz się precyzyjnie rozróżniać *narzędzia*, *zasoby* i *prompty*, aby nie kazać agentowi wykonywać akcji na statycznych plikach
4. Napiszesz od zera własną implementację serwera i klienta w Pythonie, zamiast polegać na gotowych czarnych skrzynkach
5. Zobaczysz, jak wygląda komunikacja procesów pod maską i dlaczego lokalne stdio to świetny poligon, ale koszmar na produkcji
6. Przekonasz się, dlaczego wrzucenie wszystkich dostępnych narzędzi do jednego worka prowadzi do paraliżu decyzyjnego modelu
7. Dowiesz się, w jaki sposób zwykłe opisy funkcji i typy zmiennych w Pythonie stają się precyzyjną instrukcją sterującą dla modelu językowego



## Agent zyskuje ręce

Spójrzmy na to, co do tej pory zbudowaliśmy. W module 1 zaczęliśmy od **inżynierii kontekstu** – nasz agent zaczął rozumieć, kim jest i co ma zrobić. W module 2 daliśmy mu **pamięć** – przestał być systemem z amnezją, zyskał przeszłość i ciągłość.

Ale powiedzmy sobie szczerze: do tej pory nasz agent był jak genialny filozof zamknięty w szklanym słoiku. Mógł tworzyć wspaniałe plany, pamiętać Twoje preferencje i prowadzić błyskotliwe rozmowy, ale był całkowicie odcięty od możliwości wpływania na rzeczywistość. Nie mógł wysłać e-maila, sprawdzić kalendarza ani pobrać danych z CRM.

**Moduł 3 to moment, w którym rozbijamy ten słoik.**

Dajemy agentowi "ręce". Łączymy go ze światem zewnętrznym. To jest ta chwila, w której systemy Software 3.0 stają się naprawdę użyteczne, ale też – i musimy to powiedzieć głośno – niebezpieczne.

![](../../module1/images/context_engineering.jpg)

Wszyscy ekscytują się wizją agentów z poziomu 6 lub 7, którzy "robią wszystko sami". My jednak twardo stąpamy po ziemi. Budujemy solidne systemy z poziomu 4 i 5. Dlaczego? Bo dodanie narzędzi to nie tylko nowa funkcja. To **eksplozja złożoności kontekstowej**. Pamiętasz "Czterech Jeźdźców" z lekcji 1.2? Kiedy dodasz agentowi dostęp do API, bazy danych i Internetu, ryzyko *Confusion* (zmylenia) i *Poisoning* (zatrucia) rośnie wykładniczo.

Dlatego w tym module nie będziemy po prostu "podłączać wtyczek". Będziemy projektować bezpieczne i stabilne interfejsy dla działań agenta.


## Ustalmy od razu model mentalny, według którego będziemy działać z narzędziami

Zanim napiszemy pierwszą linię kodu, musimy skalibrować nasze myślenie. W świecie narzędzi AI, zwłaszcza teraz, panuje niesamowity szum. Jako inżynierowie Software 3.0, trzymamy się twardych zasad, które chronią nas przed popełnianiem drogich błędów:

1.  Kod to obciążenie. Idealny kod to brak kodu, dlatego tam, gdzie się da, promujemy minimalizm i wykorzystanie gotowych standardów, zamiast pisania własnych "wynalazków".

2.  `MCP` to jedynie detal techniczny. Traktuj ten protokół jak złącze - często podaje się analogię do *USB-C*. Jest świetny, bo standaryzuje wtyczkę, ale ważniejszy jest problem biznesowy, który rozwiązujesz, a nie specyfikacja kabla.

3.  Produkcja to nie demo na laptopie. Odrzucamy rozwiązania oparte na lokalnym `stdio`, które działają tylko u dewelopera. Produkcja wymaga protokołów sieciowych (HTTP), autoryzacji i logowania.

4.  Bezpieczeństwo to fundament, a nie nowa funkcjonalność z fajerwerkami. *Prompt Injection*, *Data Exfiltration* ... to realne zagrożenia, gdy dajesz agentowi dostęp do danych. Happy Path nas nie interesuje – interesuje nas to, co się stanie, gdy ktoś spróbuje oszukać system.

5.  Stabilność ponad nowinki. Uczymy się wzorców architektonicznych, które przetrwają zmianę wersji biblioteki. Narzędzia się zmieniają, architektura pozostaje.

6.  Im więcej połączeń, tym większa użyteczność, ale i ryzyko. To jest dylemat, z którym będziemy pracować: izolowany agent jest bezpieczny, ale bezużyteczny. Agent połączony ze wszystkim jest super-użyteczny, ale stanowi ogromne zagrożenie. Szukamy złotego środka.

7.  Koniec ze statycznymi sekretami. W świecie agentów, którzy sami decydują o użyciu narzędzi, statyczne klucze API w kodzie to proszenie się o kłopoty. Przechodzimy na dynamiczny dostęp.

Ruszamy?

## Czym jest MCP i dlaczego teraz?

Musimy porozmawiać o **[Model Context Protocol (MCP)](https://modelcontextprotocol.io)**, bo w latach 2024-2025 ten temat zdominował dyskusje o narzędziach AI.

W skrócie: jest to otwarty standard stworzony przez Anthropic (przy wsparciu innych gigantów), który ma na celu ustandaryzowanie sposobu, w jaki modele AI łączą się z danymi i narzędziami. Zamiast pisać osobną integrację dla każdego modelu i każdej bazy danych, piszesz jeden "konektor" MCP, który działa wszędzie.

![](../images/mcp.jpg)

Spójrz na dynamikę adopcji tego standardu. To dzieje się błyskawicznie:

- **Listopad 2024:** Start standardu MCP
- **Styczeń - Marzec 2025:** Integracja z głównymi edytorami kodu ([Cursor](https://cursor.com/docs/context/mcp), [VS Code](https://code.visualstudio.com/docs/copilot/customization/mcp-servers))
- **Wiosna 2025:** Google i Microsoft dołączyły do gry, natywne wsparcie w [Windows 11](https://blogs.windows.com/windowsexperience/2025/05/19/securing-the-model-context-protocol-building-a-safer-agentic-future-on-windows/)
- **Wrzesień 2025:** Pełne wsparcie w [ChatGPT](https://help.openai.com/en/articles/12584461-developer-mode-apps-and-full-mcp-connectors-in-chatgpt-beta), narzędziach [developerskich Chrome](https://developer.chrome.com/blog/chrome-devtools-mcp)
- ...

Brzmi świetnie? Tak. Ale pojawia się tu szereg wyzwań — między innymi związanych z bezpieczeństwem.

### Niemożliwy wybór i iluzja bezpieczeństwa

Żyjemy w czasach, gdzie adopcja wyprzedza rozsądek. "MCP adoption is accelerating" – to hasło widać wszędzie. Stoimy dziś przed pozornie niemożliwym wyborem:

1.  Dać agentom szeroki dostęp do systemów i zaakceptować ryzyko wycieku danych.
2.  Ograniczyć możliwości agenta i poświęcić wartość biznesową.

Instalowanie przypadkowych serwerów MCP z GitHub i dawanie im dostępu do Twoich prywatnych plików czy baz danych, tylko dlatego, że "to działa", to rosyjska ruletka. Ale nawet jeśli zabezpieczysz ten dostęp, wpadasz w drugą pułapkę.

### Druga strona medalu: Klątwa nadmiaru i "Context Obesity"

Wyobraź sobie, że dajesz agentowi dostęp do 50 różnych narzędzi. Każde z nich ma swoją definicję, parametry i opis. Gdzie to wszystko trafia? Oczywiście – do promptu systemowego (bloku 5 z naszej architektury).

I tu pojawia się krytyczny problem inżynieryjny: **jak wybrać właściwe narzędzie, nie zapychając "pamięci operacyjnej" modelu?**

Ścieżka domyślna, którą podąża większość początkujących, wygląda tak: wrzucamy definicje wszystkich dostępnych narzędzi do kontekstu i liczymy na to, że model "sobie wybierze". To gwarancja porażki z trzech powodów:

1.  **Zmylenie (Confusion):** Im więcej narzędzi widzi model, tym trudniej mu wybrać to jedno właściwe. Zaczyna mylić parametry podobnych funkcji.
2.  **Zalew danych (Output Flooding):** Co się stanie, gdy narzędzie zadziała i zwróci wynik? Jeśli zapytanie SQL zwróci 10 tysięcy wierszy, a Ty bezrefleksyjnie wrzucisz to z powrotem do promptu, natychmiast przekroczysz limity i wywołasz amnezję u agenta (zjawisko *Lost in the Middle*).
3.  **Stabilność:** Stabilny system to taki, w którym kontekst jest minimalny i precyzyjny. Każde zbędne narzędzie w prompcie to szum, który obniża IQ Twojego agenta.

Dlatego w tym module nauczymy się strategii, jak tym zarządzać. Zamiast "rzucać workiem narzędzi" w model, będziemy budować mechanizmy, które precyzyjnie dobierają odpowiedni zestaw akcji do aktualnego zadania. To inżynieria kontekstu przeniesiona na wyższy poziom.

W module 3 pokażę Ci trzecią drogę: **bezpieczną i oszczędną orkiestrację**.

### ☝️ Iluzja potęgi (pułapka "kłębowiska")

Wielu entuzjastów widzi MCP jako magiczną różdżkę. Myślą: *"Skoro podłączenie narzędzia jest teraz tak proste, podłączę 100 serwerów MCP, dam agentowi dostęp do całego mojego komputera i on sam wszystko zrobi"*.

To myślenie życzeniowe, które prowadzi do katastrofy. Aby zrozumieć dlaczego, musimy użyć prostej, fizycznej analogii.

![](../../module1/images/software3.0_7_levels.jpg)



**Traktuj MCP jak standard kabla (np. USB-C).**
To świetnie, że mamy standard. Dzięki niemu techniczne połączenie punktu A (Twoja baza danych) z punktem B (Agent AI) jest trywialne. Wtyczka pasuje, prąd płynie.

Ale co się stanie, gdy w ten sposób "bezrefleksyjnie" podłączysz 50, 100 albo 200 narzędzi? Stworzysz coś takiego:


Brutalna prawda jest taka: **Więcej narzędzi = Więcej problemów z kontekstem.**

**To jest właśnie Twój prompt systemowy (Context Window), gdy wrzucisz do niego definicje wszystkich dostępnych narzędzi.**

Jak podchodzimy tworząc świadomie Software 3.0?
> **Nawet najpotężniejszy model LLM zgubi się w takim bałaganie. Tutaj do gry musi wejść architekt i oczywiście inteligencja naturalna*.*

Dla modelu każde narzędzie w prompcie to kolejny "kabel", który musi śledzić. Jeśli dasz mu ich zbyt wiele:
1.  **Zmylenie (Confusion):** Model nie będzie wiedział, który "kabel" wybrać do prostego zadania.
2.  **Halucynacje:** Zacznie wymyślać parametry, próbując dopasować się do szumu informacyjnego.
3.  **Awaria:** System stanie się niestabilny i nieprzewidywalny.

MCP rozwiązuje problem **łączności** (jak podłączyć), ale to Ty musisz rozwiązać problem **selekcji** (co i kiedy podłączyć). Bez Twojej inżynierii z poziomu 4/5, podłączenie agenta do Internetu skończy się cyfrowym "kłębowiskiem", a nie działającym produktem.


![](../images/too_many_tools.jpg)

Dlatego w tym module nauczymy się nie tylko jak narzędzia *podłączać*, ale przede wszystkim – jak nimi *zarządzać*, by Twój prompt wyglądał jak uporządkowana serwerownia, a nie jak to zdjęcie powyżej.

## Architektura MCP: Trzech aktorów w Twoim systemie

Zanim zaczniemy łączyć kable, musimy zrozumieć, kto w tej grze w ogóle bierze udział. Dokumentacja techniczna bywa sucha, ale w świecie **Software 3.0** te trzy pojęcia – Host, Klient i Serwer – mają fundamentalne znaczenie dla tego, jak Twój agent "myśli" i działa.

To nie jest tylko techniczny żargon. To struktura organizacyjna Twojego przyszłego systemu.

W architekturze MCP zawsze występuje trójca:

### 1. Host – czyli Twój system Software 3.0
W dokumentacji przeczytasz, że Hostem jest np. *Claude Desktop* czy *Visual Studio Code*. To prawda w świecie konsumenckim. Ale tutaj, w DataWorkshop, patrzymy na to inaczej — fundamentalnie. Nie jesteśmy tylko biernymi konsumentami — jesteśmy twórcami systemów Software 3.0, dlatego Hostem jest nasza aplikacja. To my wyznaczamy kierunek i definiujemy przyszłość, zamiast jedynie ją odtwarzać, prawda?

**Hostem jest Twoja aplikacja.** To ten kod (np. w Pythonie), który piszesz. Host to "centrala". To tutaj siedzi mózg (LLM), tutaj jest pamięć (z modułu 2) i tutaj zapadają decyzje. Host nie wykonuje pracy fizycznej – on orkiestruje. Decyduje: *"Teraz potrzebuję danych z bazy"*, *"Teraz muszę wysłać e-maila"*. Ale żeby to zrobić, potrzebuje rąk.

![](../images/mcp_software3.0.jpg)

### 2. Klient – "niedoceniany pracownik"
O kliencie mówi się najmniej, a to on odwala czarną robotę. Zwróć uwagę na kluczową zasadę architektury: **Jeden Host może mieć wielu klientów.**

Dlaczego? Ponieważ Host (Twoja aplikacja) nie łączy się z narzędziami bezpośrednio "na brudno". Host powołuje do życia **klienta MCP** dla każdego serwera, z którym chce gadać.
- Chcesz połączyć się z GitHubem? Twój Host używa *Klienta A*.
- Chcesz połączyć się z lokalnym systemem plików? Twój Host tworzy *Klienta B*.

Klient to "tłumacz". To on pilnuje protokołu, to on fizycznie wysyła zapytanie i odbiera wynik. Jest strażnikiem połączenia 1:1 z konkretnym serwerem.


### 3. Serwer  – dostawca kontekstu i zdolności
To tutaj żyją "zdolności" Twojego systemu. I tu musimy obalić popularny mit: **Serwer MCP to nie tylko "zbiór funkcji do wywołania" (tools)**.

Gdyby tak było, MCP niewiele różniłoby się od zwykłego API REST. Siła tego protokołu leży w tym, że Serwer jest pełnoprawnym partnerem w rozmowie. Co to jest? Na początek przedstawię trzy najważniejsze elementy:

1.  **Zasoby (Resources):** To pasywny dostęp do danych. Wyobraź sobie, że Serwer mówi do Hosta: *"Hej, mam tu dostęp do logów systemowych, plików konfiguracyjnych albo bieżącego stanu bazy danych. Możesz to czytać jak plik, nie musisz wywoływać skomplikowanych funkcji"*. To są oczy i uszy agenta.

2.  **Narzędzia (Tools):** To jest to, co wszyscy znają – aktywne działanie. *"Wyślij e-mail", "Zrestartuj kontener", "Zrób commit"*. To są ręce agenta.

3.  **Prompty (Prompts):** To gotowe szablony interakcji. Serwer może powiedzieć: *"Wiem, jak najlepiej mnie pytać o błędy. Użyj tego gotowego szablonu promptu, żeby przeanalizować ten log"*. To standaryzacja komunikacji.


### Jak to wygląda w praktyce?

```
╔══════════════════════════════════════════════════════════════════════════════╗
║                       MCP HOST (AI Application)                              ║
║                       e.g., Claude Desktop, Cursor IDE                       ║
║                                                                              ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │                      AI Brain / Core Logic                             │  ║
║  │                 (Reasoning, Memory, Orchestration)                     │  ║
║  │                                                                        │  ║
║  │   "I need to read a file" → finds server with that capability          │  ║
║  │   "I need to query database" → finds server with that capability       │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                    │                                         ║
║                                    ↓                                         ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │                        MCP Client Layer                                │  ║
║  │            (Generic protocol handler - spawns connections)             │  ║
║  │                                                                        │  ║
║  │   ┌────────────┐       ┌────────────┐       ┌────────────┐             │  ║
║  │   │ Connection │       │ Connection │       │ Connection │             │  ║
║  │   │ Instance 1 │       │ Instance 2 │       │ Instance 3 │             │  ║
║  │   │            │       │            │       │            │             │  ║
║  │   │  1:1 with  │       │  1:1 with  │       │  1:1 with  │             │  ║
║  │   │  Server 1  │       │  Server 2  │       │  Server 3  │             │  ║
║  │   └────────────┘       └────────────┘       └────────────┘             │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
╚═══════════════╪═══════════════════════╪═══════════════════════╪══════════════╝
                │                       │                       │
                │ HTTP                  │ HTTP                  │ HTTP
                │                       │                       │
                ↓                       ↓                       ↓
                                                                
           ☁️ CLOUD                ☁️ CLOUD                ☁️ CLOUD
                                                                
    ┌────────────────────┐  ┌────────────────────┐  ┌────────────────────┐
    │    MCP Server 1    │  │    MCP Server 2    │  │    MCP Server 3    │
    │                    │  │                    │  │                    │
    │    GitHub API      │  │   Stripe Payments  │  │  Supabase DB       │
    │                    │  │                    │  │                    │
    │  Exposes Tools:    │  │  Exposes Tools:    │  │  Exposes Tools:    │
    │  ┌──────────────┐  │  │  ┌──────────────┐  │  │  ┌──────────────┐  │
    │  │ • list_repos │  │  │  │ • charge     │  │  │  │ • query      │  │
    │  │ • create_pr  │  │  │  │ • refund     │  │  │  │ • insert     │  │
    │  │ • get_issues │  │  │  │ • subscribe  │  │  │  │ • update     │  │
    │  └──────────────┘  │  │  └──────────────┘  │  │  └──────────────┘  │
    │                    │  │                    │  │                    │
    │  Exposes Resources:│  │  Exposes Resources:│  │  Exposes Resources:│
    │  ┌──────────────┐  │  │  ┌──────────────┐  │  │  ┌──────────────┐  │
    │  │ • repo://    │  │  │  │ • customer://│  │  │  │ • table://   │  │
    │  │ • pr://      │  │  │  │ • invoice:// │  │  │  │ • schema://  │  │
    │  └──────────────┘  │  │  └──────────────┘  │  │  └──────────────┘  │
    └────────────────────┘  └────────────────────┘  └────────────────────┘
       [REMOTE - hosted]       [REMOTE - hosted]       [REMOTE - hosted]
       
       
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
                                                                
    ⚠️  EXCEPTION: Local servers (require installation & configuration)
                                                                
    ┌────────────────────┐                                      
    │    MCP Server      │   • Requires technical knowledge     
    │                    │   • User must install & maintain     
    │    Filesystem      │   • Use only when necessary          
    │    (stdio)         │     (e.g., local file access)        
    └────────────────────┘                                      
         [LOCAL]
```

## Jak oni ze sobą "rozmawiają"? (warstwa transportowa)

Skoro mamy już aktorów (Host, Klient, Serwer), musimy ustalić, w jaki sposób przesyłają między sobą informacje. W świecie MCP wyróżniamy trzy główne standardy transportu:

1.  **Stdio** (Standard Input/Output)
2.  **SSE** (Server-Sent Events)
3.  **HTTP z JSON-RPC** (często nazywany Streamable HTTP)

Każdy z nich ma swoje miejsce w historii i konkretne zastosowanie. Przejdźmy przez nie krok po kroku, aby zrozumieć, dlaczego ewolucja poszła w konkretnym kierunku.

### 1. `Stdio` – "Działa na moim komputerze" (lokalnie)
To rozwiązanie, które zobaczysz w większości tutoriali i dem. Działa to tak, jakbyś połączył dwa programy na jednym komputerze "rurą". Host uruchamia proces Serwera w tle i komunikują się przez konsolę (standardowe wejście i wyjście).

- **Zaleta:** Jest szybkie i bezpieczne (dane nigdy nie opuszczają Twojego komputera. Owszem, serwer może komunikować się ze światem zewnętrznym, ale połączenie między klientem a serwerem odbywa się lokalnie — na tym samym urządzeniu). Idealne dla dewelopera (albo kiedy budujemy serwer MCP i chcemy go potestować — pokażę przykład w tej lekcji).
*   **Wada (inżynierski ból głowy):** Wdrożenie tego w dużej skali to koszmar. Wymaga, aby każdy użytkownik miał na swoim komputerze zainstalowane odpowiednie środowisko (Python, Node.js) i zależności. Jeśli coś działa u Ciebie, nie ma gwarancji, że zadziała u klienta.

### 2. `SSE` (Server-Sent Events) – ślepa uliczka ewolucji

Kiedy wychodzimy do sieci, pierwszym naturalnym pomysłem było SSE. Znasz to doskonale z ChatGPT – zadajesz pytanie, a odpowiedź pojawia się na ekranie słowo po słowie. To jest klasyczny streaming.

W tym scenariuszu to działa świetnie, bo relacja jest prosta: **Jeden mówi, drugi słucha.**

Jednak w MCP szybko okazało się to architektonicznym błędem. Dlaczego?
Bo MCP to nie jest monolog, to **dialog**. `MCP Serwer` może chcieć w dowolnym momencie zapytać o coś Hosta (np. przy mechanizmie *Sampling*).

#### Problem "dwóch Mostów"
SSE z definicji działa w jedną stronę (Serwer -> Klient). Aby umożliwić Klientowi (Hostowi) odpowiedź, inżynierowie musieli budować obok drugie, niezależne połączenie HTTP (POST endpoint), żeby wysyłać wiadomości w drugą stronę.

Wyobraź sobie, że budujesz autostradę.
Zamiast zbudować jedną szeroką drogę, po której ruch odbywa się w obie strony, budujesz dwa oddzielne mosty nad rzeką, oddalone od siebie o kilometr.
- Mostem A (SSE) płyną informacje od MCP Serwera
- Mostem B (HTTP POST) płyną informacje od Klienta

Działa? Tak. 

Ale jest to skomplikowane w utrzymaniu, trudne w debugowaniu i po prostu nieefektywne. Utrzymywanie dwóch fizycznych połączeń do obsługi jednej logicznej rozmowy to narzut, którego chcemy unikać. Dlatego branża od tego odchodzi.

```
╔══════════════════════════════════════════════════════════════════════════════╗
║                    ❌ SSE APPROACH (Deprecated)                              ║
║                    "Two Bridges Problem"                                     ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║   ┌─────────────────────┐                      ┌─────────────────────┐       ║
║   │     MCP CLIENT      │                      │     MCP SERVER      │       ║
║   │     (in Host)       │                      │     (Remote)        │       ║
║   └─────────────────────┘                      └─────────────────────┘       ║
║              │                                            │                  ║
║              │                                            │                  ║
║              │    ┌────────────────────────────────┐      │                  ║
║              │    │  🌊 RIVER (Network boundary)   │      │                  ║
║              │    └────────────────────────────────┘      │                  ║
║              │                                            │                  ║
║              │         BRIDGE A: /sse endpoint            │                  ║
║              │  ┌─────────────────────────────────────┐   │                  ║
║              │  │◄────────────────────────────────────────┤ Server → Client  ║
║              │  │        SSE (one-way only)           │   │ (responses,      ║
║              │  │        Server-Sent Events           │   │  streaming)      ║
║              │  └─────────────────────────────────────┘   │                  ║
║              │                                            │                  ║
║              │              🚧 GAP 🚧                     │                  ║
║              │                                            │                  ║
║              │         BRIDGE B: /sse/messages            │                  ║
║              │  ┌─────────────────────────────────────┐   │                  ║
║              ├───────────────────────────────────────►│ Client → Server      ║
║                 │        HTTP POST (separate!)        │    (requests,        ║
║                 │        Different connection         │     commands)        ║
║                 └─────────────────────────────────────┘                      ║
║                                                                              ║
║   Problems:                                                                  ║
║   ┌────────────────────────────────────────────────────────────────────┐     ║
║   │ • Two connections to maintain for ONE logical conversation         │     ║
║   │ • Complex session correlation between bridges                      │     ║
║   │ • If Bridge A drops → responses lost, no recovery                  │     ║
║   │ • Hard to debug (which bridge failed?)                             │     ║
║   │ • More infrastructure, more failure points                         │     ║
║   └────────────────────────────────────────────────────────────────────┘     ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝

```


### 3. HTTP z JSON-RPC – standard Produkcyjny

Dlatego docelowym standardem, w którego stronę zmierza cała branża, jest **Streamable HTTP**.

Działa to jak **klasyczna rozmowa telefoniczna** albo nowoczesna autostrada.
Mamy jeden kanał (jedno połączenie), w ramach którego obie strony mogą inicjować wymianę informacji.
*   Nie ma potrzeby budowania "drugiego mostu".
*   Opieramy się na standardzie, na którym stoi cały Internet.
*   Skalowalność, autoryzacja (OAuth, API Keys) i load balancing działają tak, jak w każdym innym serwisie webowym.

Podsumowując: Choć SSE miało swoje "pięć minut" w historii MCP, to przyszłość należy do pełnej, dwukierunkowej komunikacji po HTTP. To jedyny sposób, by budować systemy, które są stabilne i łatwe w utrzymaniu na dużą skalę.


Stdio jest świetne do lokalnych plików i testów. SSE było dobre dla prostych chatbotów. Ale **Software 3.0 budujemy na HTTP**, bo tylko tam możliwa jest prawdziwa, partnerska współpraca między Agentem a jego Narzędziami.


```

╔══════════════════════════════════════════════════════════════════════════════╗
║                    ✅ STREAMABLE HTTP (Current Standard)                     ║
║                    "Single Highway"                                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║   ┌─────────────────────┐                      ┌─────────────────────┐       ║
║   │     MCP CLIENT      │                      │     MCP SERVER      │       ║
║   │     (in Host)       │                      │     (Remote)        │       ║
║   └─────────────────────┘                      └─────────────────────┘       ║
║              │                                            │                  ║
║              │                                            │                  ║
║              │    ┌────────────────────────────────────┐  │                  ║
║              │    │  🌊 RIVER (Network boundary)       │  │                  ║
║              │    └────────────────────────────────────┘  │                  ║
║              │                                            │                  ║
║              │      SINGLE BRIDGE: /mcp endpoint          │                  ║
║              │  ╔═════════════════════════════════════╗  │                   ║
║              ├══╣ ◄───────────────────────────────────╣ Bidirectional        ║
║              ├══╣ ───────────────────────────────────►╣ communication        ║
║                 ║      Streamable HTTP (both ways)    ║    on ONE            ║
║                 ║      Same connection, same channel  ║    connection        ║
║                 ╚═════════════════════════════════════╝                      ║
║                                                                              ║
║   Benefits:                                                                  ║
║   ┌────────────────────────────────────────────────────────────────────┐     ║
║   │ ✓ One connection = one conversation (simple!)                      │     ║
║   │ ✓ Dynamic: simple request/response OR streaming when needed        │     ║
║   │ ✓ True bidirectional: Server can ask Client questions (Sampling)   │     ║
║   │ ✓ Easier error handling (one channel for everything)               │     ║
║   │ ✓ Future: resumability, cancellation, session management           │     ║
║   └────────────────────────────────────────────────────────────────────┘     ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝

```




### 4. WebSockety – kość niezgody

W tym momencie Twój inżynierski instynkt pewnie krzyczy: *"Zaraz! Skoro chcemy pełnej, dwukierunkowej komunikacji, dlaczego po prostu nie użyjemy WebSocketów? Przecież one do tego służą!"*.

I masz rację. To był (i jest) temat najbardziej **burzliwych dyskusji** w społeczności MCP. Wielu inżynierów sieciowych łapało się za głowy, widząc, że MCP wymyśla koło na nowo ("Streamable HTTP"), zamiast wziąć gotowy standard.

Dlaczego więc MCP "obraziło się" na WebSockety? Decyzja była pragmatyczna, a nie polityczna. Chodziło o **"szklany sufit przeglądarki"** i realia korporacyjne.

**Gdzie WebSockety poległy w tej bitwie?**
1.  **Problem Pierwszego Uścisku Dłoni (Handshake):** W przeglądarce, kiedy inicjujesz połączenie WebSocket, masz bardzo ograniczone możliwości dodawania niestandardowych nagłówków (np. `Authorization: Bearer ...`). To brzmi jak błahostka, ale w systemach bezpieczeństwa to showstopper. MCP chce, żeby autoryzacja była tak prosta jak w zwykłym REST API.
2.  **Koszmar Infrastruktury (Middleboxes):** Korporacyjne firewalle, proxy i load balancery *nienawidzą* długo żyjących połączeń. Często zrywają je po 60 sekundach ciszy. Utrzymanie stabilnego WebSocketa w banku czy dużej korporacji wymaga często specjalnej konfiguracji sieci. Zwykłe HTTP (POST/GET) przechodzi wszędzie.
3.  **Model Mentalny:** WebSockety są "stanowe" (Stateful). Wymagają ciągłego połączenia. Model HTTP jest "bezstanowy" (Stateless). MCP chciało standardu, który jest łatwiejszy do skalowania (np. na Serverless/Lambda), gdzie trzymanie otwartego WebSocketa jest drogie lub niemożliwe.

#### Kompromis: Streamable HTTP
Dlatego wybrano "Streamable HTTP". To technologia, która **udaje WebSockety**, używając zwykłego HTTP.
*   Zachowuje się jak zwykła strona WWW (łatwo przechodzi przez firewalle).
*   Pozwala na łatwą autoryzację.
*   Ale pod spodem pozwala na dwukierunkową wymianę danych.

#### Czy WebSockety są zakazane?
Nie! I to jest ważne. Specyfikacja MCP nie zabrania użycia WebSocketów "pod maską" jako warstwy transportowej, jeśli masz pełną kontrolę nad infrastrukturą (np. wewnątrz własnego klastra). Ale jako **domyślny standard publiczny**, wygrało nudne, ale niezawodne HTTP, zamiast idealnych, ale problematycznych WebSocketów.

![](../images/mcp_analogy.jpg)

To jest moment, w którym teoria zamienia się w praktykę. Przechodzimy od rysowania schematów do pisania kodu.

W tej sekcji zrealizujemy zadanie **3.1.1**, które ma podwójny cel:
1.  Pokazać Ci, jak łatwo postawić własny serwer MCP, używając oficjalnego SDK.
2.  Zdemistyfikować rolę "Hosta" – napiszesz prosty skrypt klienta, który "udaje" zaawansowanego agenta (lub Claude Desktop).

Zanim zaczniemy, musimy ustalić pewną filozofię pracy z narzędziami.

## Oficjalne SDK a "grzebanie pod maską"

MCP to otwarty standard, a organizacja za nim stojąca (pod przewodnictwem Anthropic) dba o to, aby nie trzeba było ręcznie parsować ramek `JSON-RPC`. Przygotowali gotowe **SDK (Software Development Kits)** dla najpopularniejszych języków.

Będziemy korzystać z **[oficjalnego Python SDK](https://modelcontextprotocol.io/docs/sdk)** (`mcp`).

Dlaczego? Czy to oznacza, że idziemy na skróty?
I tak, i nie. W DataWorkshop wyznajemy zasadę **"świadomej abstrakcji"**.
*   **Edukacyjnie:** Chcemy wiedzieć, co dzieje się na najniższym poziomie (i za moment zajrzymy głęboko pod maskę, analizując surowe pakiety JSON przesyłane między procesami).
*   **Produkcyjnie:** Nie chcemy wymyślać koła na nowo. Protokół ewoluuje, zmieniają się detale implementacyjne transportu. Użycie SDK daje nam stabilność – to twórcy standardu martwią się o zgodność z najnowszą wersją specyfikacji, a nie my.

Dzięki temu SDK, zmiana transportu z `stdio` (lokalnie) na `HTTP` (zdalnie) będzie w przyszłości kwestią **zmiany jednej linijki konfiguracji**, a nie przepisywaniem całego serwera.

## Dlaczego uczymy się obu stron (Klient i Serwer)?

Możesz zapytać: *"Jestem inżynierem AI, po co mam pisać kod serwera? Przecież dostanę gotowe API od backendowców"*. Albo odwrotnie: *"Jestem backendowcem, po co mi kod klienta agenta?"*.

W świecie Software 3.0 te granice się zacierają. Wyznajemy zasadę **Full-Stack AI Engineering**:
1.  **Zrozumienie kontraktu:** Aby napisać dobrego Agenta (Klienta), musisz wiedzieć, jak "myśli" narzędzie po drugiej stronie. Musisz czuć, jak definiuje się schematy danych, żeby wiedzieć, jakich błędów unikać w promptach.
2.  **Debugowanie:** Gdy system na produkcji przestanie działać (a przestanie), musisz umieć sprawdzić, czy wina leży po stronie "mózgu" (LLM źle wybrał narzędzie i parametry), czy "rąk" (Serwer MCP źle wykonał logikę lub rzucił błędem). Bez znajomości obu stron będziesz błądzić po omacku.
3.  **Szybkie Prototypowanie:** Najlepsze systemy powstają, gdy jedna osoba potrafi szybko postawić prostą "zaślepkę" serwera i natychmiast przetestować ją z agentem.

## Ważna uwaga o transporcie (stdio vs HTTP)

W poprzedniej sekcji mocno argumentowaliśmy, że przyszłość to `HTTP`, a `stdio` ma swoje wady. Dlaczego więc w tym pierwszym zadaniu użyjemy `stdio`?

Z czystego pragmatyzmu. To **najszybsza droga do "Hello World"**.
Nie musisz konfigurować serwera webowego, otwierać portów, generować kluczy API ani martwić się o firewalle. Uruchamiasz dwa procesy na jednej maszynie i one po prostu "gadają".

Traktuj `stdio` jako swoje **środowisko deweloperskie (localhost)** – idealne do nauki i testów. Kiedy już upewnimy się, że logika działa, przełączenie się na `HTTP` (środowisko produkcyjne) będzie trywialne właśnie dzięki użyciu SDK.

No to zbudujmy Twój pierwszy system MCP (serwer i klient).


## Jak budować Serwery MCP? (krok po kroku)

Zanim rzucę Cię na głęboką wodę (co i tak nastąpi, ale z pełnym szacunkiem do Twojego tempa), zobaczmy wspólnie, jak wygląda anatomia serwera MCP. To nie jest czarna magia – to po prostu definiowanie funkcji w Pythonie, które "dekorujemy" w specjalny sposób, aby Agent mógł je zobaczyć i zrozumieć.

### Przykład: "Hello Server"

Stwórzmy prosty serwer, który posiada trzy fundamentalne zdolności:
1.  **Narzędzie (Tool):** Potrafi powtórzyć tekst (echo).
2.  **Zasób (Resource):** Udostępnia statyczny tekst (np. powitanie).
3.  **Prompt:** Udostępnia szablon rozmowy.


Przyjrzyj się uważnie poniższemu kodowi. Każda linijka ma znaczenie.

In [1]:
%%writefile server_hello.py 
#☝️ This is a Jupyter "magic command"

from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base

# FastMCP is the high-level, beginner-friendly way to create MCP servers
# Think of it like Flask/FastAPI but for MCP protocol
mcp = FastMCP("Hello Server")


@mcp.tool(name="echo_tool")
def echo_tool(text: str) -> str:
    return f"Echo: {text}"


@mcp.resource("greeting://hello")
def get_greeting() -> str:
    """Returns a static greeting message."""
    return "Hello! This is a static resource from MCP server."


@mcp.prompt("simple_prompt")
def simple_prompt(user_name: str) -> list[base.Message]:
    """Generates a welcome message for a user."""
    return [
        base.UserMessage(f"Hello, my name is {user_name}"),
        base.AssistantMessage(f"Welcome {user_name}! How can I help you today?")
    ]


if __name__ == "__main__":
    mcp.run()

Overwriting server_hello.py


## Analiza kodu – co tu się właściwie stało?

Rozłóżmy ten kod na czynniki pierwsze. Jeśli nigdy wcześniej tego nie robiłeś, te 5 kroków jest kluczem do zrozumienia całej magii MCP.

### 1. Inicjalizacja (`mcp = FastMCP(...)`)
Tworzymy instancję serwera. Ciąg znaków `"Hello Server"` to nie tylko nazwa dla Ciebie. Ta nazwa zostanie przesłana do Klienta (Agenta) podczas pierwszego połączenia (handshake). Dzięki temu Agent wie, z kim rozmawia. Używamy biblioteki `FastMCP`, która ukrywa przed nami całą złożoność protokołu JSON-RPC.

### 2. Definiowanie Narzędzi (`@mcp.tool`)
To są "ręce" agenta. Używamy dekoratora, aby zarejestrować funkcję jako narzędzie.
*   **Krytycznie ważne:** Zwróć uwagę na **type hints** (`text: str`) oraz **docstring** (opis funkcji w potrójnym cudzysłowie).
*   Dlaczego? Bo MCP automatycznie czyta te elementy i konwertuje je na schemat JSON, który jest wysyłany do LLM-a. Jeśli nie opiszesz funkcji w docstringu, model nie będzie wiedział, do czego ona służy ani jak jej użyć!

### 3. Definiowanie Zasobów (`@mcp.resource`)
To są "oczy" agenta. Zasoby to pasywny dostęp do danych.
*   Zauważ dziwny ciąg: `"greeting://hello"`. To jest **URI** (Uniform Resource Identifier).
*   Działa to jak adres strony internetowej lub ścieżka do pliku. Agent może "poprosić" o treść znajdującą się pod tym adresem. W tym przypadku, gdy agent odpyta o `greeting://hello`, uruchomi się nasza funkcja i zwróci tekst.

### 4. Definiowanie Promptów (`@mcp.prompt`)
To jest "know-how" serwera. Zamiast kazać Agentowi wymyślać, jak zacząć rozmowę, serwer daje mu gotowy scenariusz.
*   Funkcja `simple_prompt` nie zwraca zwykłego tekstu, ale **listę wiadomości** (`base.UserMessage`, `base.AssistantMessage`).
*   To pozwala ustawić "stan początkowy" rozmowy w pamięci LLM-a, np. symulując, że użytkownik się przedstawił, a asystent już mu odpowiedział.


```
┌─────────────────────────────────────────────────────────────────┐
│                    MCP SERVER: "Hello Server"                   │
│                                                                 │
│   ┌─────────────────────────────────────────────────────────┐   │
│   │                    CAPABILITIES                         │   │
│   ├─────────────────────────────────────────────────────────┤   │
│   │                                                         │   │
│   │   🔧 TOOL: echo_tool                                    │   │
│   │      └─ AI can call this to echo text back              │   │
│   │                                                         │   │
│   │   📄 RESOURCE: greeting://hello                         │   │
│   │      └─ AI can read this static greeting message        │   │
│   │                                                         │   │
│   │   💬 PROMPT: simple_prompt                              │   │
│   │      └─ Pre-built conversation template for AI          │   │
│   │                                                         │   │
│   └─────────────────────────────────────────────────────────┘   │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### 5. Uruchomienie (`mcp.run()`)
To serce całej operacji. Ta metoda przejmuje kontrolę nad procesem.
- Domyślnie nasłuchuje ona na **standardowym wejściu (stdin)** i wypisuje odpowiedzi na **standardowe wyjście (stdout)**.
- To właśnie dzięki temu możemy połączyć klienta i serwer "rurą", bez konieczności stawiania serwera HTTP, otwierania portów czy konfiguracji sieci.

### Kompas Architekta: narzędzia, zasoby i prompty

Zanim zaczniesz pisać własny serwer, musisz zrozumieć różnicę między trzema klockami, z których będziesz budować. Wiele osób myli Narzędzia z Zasobami. Oto zestawienie, które raz na zawsze rozwieje wątpliwości.

| Cecha | 🔧 TOOL (narzędzie) | 📄 RESOURCE (zasób) | 💬 PROMPT (szablon) |
| :--- | :--- | :--- | :--- |
| **Do czego służy?** | **Wykonaj akcję** (często zmienia stan świata). | **Przeczytaj dane** (pasywny dostęp do informacji). | **Ustaw kontekst** (gotowy scenariusz rozmowy). |
| **Kierunek** | Agent → Serwer (Agent "klika przycisk"). | Agent ← Serwer (Agent "czyta kartkę"). | Serwer → Agent (Serwer daje "instrukcję"). |
| **Przykłady** | "Wyślij e-mail", "Oblicz podatek", "Zrestartuj usługę". | "Treść pliku logów", "Stan konta w bazie", "Konfiguracja systemu". | "Zrób Code Review tego kodu", "Wytłumacz błąd jak juniorowi". |
| **Dekorator** | `@mcp.tool()` | `@mcp.resource()` | `@mcp.prompt()` |
| **Co zwraca?** | Dowolny wynik (tekst, JSON, status). | Treść (tekst lub bajty). | Listę wiadomości (`UserMessage`, `SystemMessage`). |
| **Kiedy użyć?** | Gdy Agent musi coś **zrobić** lub przetworzyć. | Gdy Agent musi coś **wiedzieć** (kontekst). | Gdy chcesz ustandaryzować **sposób** pracy Agenta. |

---

### Jak to rozumieć w praktyce? 

Żeby to stało się absolutnie oczywiste, wyobraź sobie Agenta jako nowego pracownika w Twoim biurze:

#### 1. TOOL (narzędzie) 
To są polecenia, które wydajesz pracownikowi.
- *Przykład:* Nie chcesz, żeby pracownik tylko "patrzył" na kalkulator. Chcesz, żeby **obliczył** wynik. Nie chcesz, żeby "patrzył" na klienta pocztowego. Chcesz, żeby **wysłał** wiadomość.
- *Klucz:* Tu często zachodzą zmiany (side-effects). Wysłanie maila nie da się cofnąć.

#### 2. RESOURCE (zasób) 
To są dokumenty, które kładziesz mu na biurku.
- *Przykład:* Pracownik musi napisać raport. Żeby to zrobić, musi **przeczytać** (Resource) logi z ostatniego tygodnia. On ich nie zmienia, on je tylko konsumuje, żeby mieć wiedzę potrzebną do wykonania zadania.
- *Klucz:* Zasoby są adresowane jak strony internetowe (URI), np. `logs://error-log`. Agent mówi: "Pokaż mi, co jest pod tym adresem".

#### 3. PROMPT (szablon) 
To jest instrukcja BHP albo procedura operacyjna.
- *Przykład:* Zamiast za każdym razem tłumaczyć pracownikowi: *"Słuchaj, jak analizujesz błąd, to najpierw sprawdź datę, potem moduł, a na końcu podaj rozwiązanie"*, dajesz mu gotowy formularz (Prompt) o nazwie `analyze_error`.
- *Klucz:* To sposób na to, by Serwer (czyli Ty, twórca narzędzia) narzucił Agentowi najlepszy sposób korzystania z Twoich usług.

## Anatomia klienta MCP

Serwer stoi. Ale żeby ożył, potrzebuje MCP Klienta.
Pisanie klienta MCP może wydawać się skomplikowane, bo zwykle też wchodzimy w świat asynchroniczności (`async/await`) i zarządzania procesami.

Ale spokojnie. Spójrz na to z lotu ptaka. Cały klient sprowadza się do **trzech logicznych kroków**:

```python
# 1. Skonfiguruj (powiedz, gdzie jest serwer)
params = StdioServerParameters(command="python", args=["server_hello.py"])

# 2. Połącz (otwórz rurę/kabel)
async with stdio_client(params) as (read, write):
    
    # 3. Rozmawiaj (używaj narzędzi)
    async with ClientSession(read, write) as session:
        await session.initialize()
        await session.call_tool("echo_tool", ...)
```


```text
┌───────────────────────┐              ┌─────────────────────────┐
│   CLIENT (Twój kod)   │              │  SERVER (server_hello)  │
└──────────┬────────────┘              └────────────┬────────────┘
           │                                        │
           │  1. Spawn Process (python server.py)   │
           │ ─────────────────────────────────────> │
           │                                        │
           │  2. Handshake (initialize)             │
           │ ─────────────────────────────────────> │
           │ <───────────────────────────────────── │
           │                                        │
           │  3. Call Tool ("echo_tool")            │
           │ ─────────────────────────────────────> │ 🔧 Uruchamia funkcję
           │                                        │    echo_tool()
           │ <───────────────────────────────────── │    Zwraca wynik
           │    Wynik: "Echo: Hello World"          │
           │                                        │
           │  4. Read Resource ("greeting://hello") │
           │ ─────────────────────────────────────> │ 📄 Czyta dane
           │ <───────────────────────────────────── │    Zwraca treść
           │    Wynik: "Hello! ..."                 │
           │                                        │
```           

To wszystko. Reszta to tylko "boilerplate" (niezbędna otoczka) Pythona.         

In [2]:
%%writefile client_hello.py

import asyncio
from mcp import ClientSession
from mcp.client.stdio import stdio_client, StdioServerParameters

async def run_client():
    # KROK 1: Konfiguracja parametrów serwera
    # Tutaj definiujemy, jak uruchomić nasz serwer.
    # W przypadku stdio, klient po prostu odpala skrypt serwera jako podproces.
    server_params = StdioServerParameters(
        command="python",           # Program do uruchomienia
        args=["server_hello.py"],   # Argumenty (nasz plik serwera)
        env=None                    # Zmienne środowiskowe (opcjonalne)
    )

    # stdio_client zarządza wejściem/wyjściem (stdin/stdout).
    # read = strumień danych OD serwera
    # write = strumień danych DO serwera
    async with stdio_client(server_params) as (read, write):
        
        # Uruchomienie sesji MCP (protocol layer)
        # ClientSession to "centrum dowodzenia", który zamienia surowe dane w obiekty MCP.
        async with ClientSession(read, write) as session:
            
            # Handshake - kluczowy moment!
            # Klient i Serwer wymieniają się informacjami o swoich możliwościach.
            await session.initialize()
            
            # --- TERAZ ZACZYNA SIĘ ZABAWA 🕺💃 ---
            
            # A. Odkrywanie narzędzi (Discovery)
            # Pytamy serwer: "Co potrafisz robić?"
            tools = await session.list_tools()
            print(f"Znalezione narzędzia: {[t.name for t in tools.tools]}")
            
            # B. Użycie narzędzia (Execution)
            # Mówimy: "Uruchom echo_tool z parametrem text='Hello World'"
            result = await session.call_tool(
                "echo_tool", 
                arguments={"text": "Hello World"}
            )
            # Serwer wykonuje kod i zwraca wynik
            print(f"Wynik narzędzia: {result.content[0].text}")

            # C. Czytanie zasobów 
            # Mówimy: "Pokaż mi zawartość pod adresem greeting://hello"
            resource = await session.read_resource("greeting://hello")
            print(f"Treść zasobu: {resource.contents[0].text}")

# Punkt wejścia programu 
if __name__ == "__main__":
    asyncio.run(run_client())

Writing client_hello.py


## Analiza – co tu się właściwie stało?


### 1. Transport (`stdio_client`)
To jest nasz "kabel". W tym przypadku (stdio), klient fizycznie uruchamia proces serwera na Twoim komputerze. To tak, jakbyś wpisał w terminalu `python server_hello.py`, ale zamiast widzieć wynik na ekranie, przechwytuje go Twój skrypt klienta.

**Dlaczego to jest ważne?**
Gdybyśmy chcieli połączyć się ze zdalnym serwerem, zmienilibyśmy tylko ten fragment na `sse_client(url="http://...")`. Reszta kodu (Krok 3) pozostałaby bez zmian!

### 2. Sesja (`ClientSession`)
To jest "tłumacz". Transport przesyła surowe bajty. Sesja dba o to, żeby te bajty układały się w poprawne komunikaty protokołu MCP (JSON-RPC). To ona pilnuje, żeby odpowiedź na Twoje pytanie trafiła do Ciebie, a nie gdzie indziej.

### 3. Interakcja (`list_tools`, `call_tool`)
To jest "użyteczność". Zauważ, że klient **nie wie**, jak zaimplementowane jest `echo_tool`. On tylko wie, że takie narzędzie istnieje i jakich argumentów wymaga. To jest esencja separacji: Agent (Klient) prosi, Serwer wykonuje.

## Uruchamiamy system

Mamy serwer (`server_hello.py`). Mamy klienta (`client_hello.py`). Oba pliki leżą już na Twoim dysku. Teraz nadszedł czas na **moment prawdy**.

Uruchomimy skrypt klienta bezpośrednio z poziomu notatnika. Użyjemy do tego znaku `!`, który w Jupyter Notebook pozwala wykonywać polecenia systemowe – to tak, jakbyśmy wpisali komendę prosto do terminala.

In [3]:
!python client_hello.py

[12/25/25 14:00:18] INFO     Processing request of type            ]8;id=560237;file:///opt/conda/envs/python310/lib/python3.10/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=251314;file:///opt/conda/envs/python310/lib/python3.10/site-packages/mcp/server/lowlevel/server.py#674\674]8;;\
                             ListToolsRequest                                   
Znalezione narzędzia: ['echo_tool']
                    INFO     Processing request of type            ]8;id=324367;file:///opt/conda/envs/python310/lib/python3.10/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=634442;file:///opt/conda/envs/python310/lib/python3.10/site-packages/mcp/server/lowlevel/server.py#674\674]8;;\
                             CallToolRequest                                    
Wynik narzędzia: Echo: Hello World
                    INFO     Processing request of type            ]8;id=527193;file:///opt/conda/envs/python310/lib/python3.10/site-package

## Co tu się właściwie wydarzyło? 🤔

Jeśli na ekranie pojawiła się lista narzędzi i odpowiedź "Echo: Hello MCP!", to znaczy, że Twój pierwszy system MCP żyje i działa!

Warto jednak zatrzymać się na chwilę i zrozumieć, że to nie było zwykłe wywołanie funkcji w Pythonie. Nie zrobiliśmy `import server`, żeby wywołać jego metodę. To była symulacja pełnoprawnej architektury rozproszonej. Oto co stało się w ułamku sekundy pod maską:

1.  Wszystko zaczyna się w momencie, gdy Jupyter uruchamia główny proces Pythona wykonujący Twój plik `client_hello.py`.
2.  Twój skrypt klienta, korzystając z konfiguracji `StdioServerParameters`, po cichu powołuje do życia **drugi, całkowicie niezależny proces** systemowy, który uruchamia `server_hello.py`.
3.  System operacyjny łączy te dwa procesy niewidzialnymi "kablami" (strumieniami stdin/stdout), tworząc zamknięty kanał komunikacyjny.
4.  Następuje "uścisk dłoni" (Handshake), w którym Klient przedstawia się jako zgodny z protokołem MCP, a Serwer odpowiada listą swoich możliwości, mówiąc: "Cześć, mam narzędzie echo i jeden zasób powitalny".
5.  Kiedy w kodzie wywołujesz `call_tool`, Klient pakuje Twoje zapytanie do formatu JSON-RPC, wysyła je rurą do Serwera, a Serwer wykonuje funkcję w swojej pamięci i odsyła wynik tą samą drogą.
6.  Na samym końcu, gdy skrypt klienta kończy pracę, automatycznie "sprząta" po sobie, zamykając proces serwera, aby nie zużywał zasobów.
7. To jest właśnie ta **bezpieczna orkiestracja**. Twój Klient (Agent) nie musi znać kodu Serwera ani nawet wiedzieć, w jakim języku został on napisany. On tylko musi wiedzieć, jak się z nim połączyć i o co może zapytać.

Teraz, gdy rozumiesz już budowę serwera, czas stworzyć coś własnego, prawda?

## 🧠 Zadanie 3.1.1: MCP server systemu zamówień (Serwer)

Twoim zadaniem jest stworzenie pliku `my_server.py`, będzie zrobić MCP serwer do systemu zamówień. Agent będzie używał tego serwera, aby sprawdzać statusy zamówień i czytać logi.

**Wymagania:**
1.  Stwórz serwer o nazwie "Order System".
2.  **Tool `get_order_status`:**
    - Przyjmuje `order_id` (str).
    - Symuluje bazę danych (np. słownik): "ORD-123" -> "SHIPPED", "ORD-456" -> "PENDING".
    - Zwraca status lub "NOT_FOUND".
3.  **Resource `system://logs`:**
    - Zwraca statyczny tekst symulujący logi, np. "System logs: All services operational."
4.  **Prompt `refund_policy`:**
    - Przyjmuje `customer_name`.
    - Zwraca wiadomość wyjaśniającą zasady zwrotów dla tego klienta.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈
    </summary>
<p>

- Skopiuj strukturę z `server_hello.py`.
- W funkcji `get_order_status` użyj prostego słownika `mock_db = {"ORD-123": "SHIPPED"}` i metody `.get()`.
- Pamiętaj o importach z `mcp.server.fastmcp`.

</p>
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć rozwiązanie 👈
    </summary>
<p>

```python
%%writefile my_server.py
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base

# Initialize the MCP Server wrapping the Order Management System
mcp = FastMCP("Order System")

# --- TOOLS (Actions) ---

@mcp.tool(description="Check the status of an order by its ID.")
def get_order_status(order_id: str) -> dict:
    """
    Retrieves the current status of an order.
    
    Args:
        order_id: The unique identifier of the order (e.g., ORD-123).
    """
    # Simulation of a DB lookup
    mock_db = {
        "ORD-123": "SHIPPED",
        "ORD-456": "PENDING",
        "ORD-999": "CANCELLED"
    }
    
    status = mock_db.get(order_id, "NOT_FOUND")
    return {"order_id": order_id, "status": status}

# --- RESOURCES (Data access) ---

@mcp.resource("system://logs")
def get_system_logs() -> str:
    """Read the latest system logs."""
    return "System logs: No critical errors today. All services operational."

# --- PROMPTS (Context templates) ---

@mcp.prompt("refund_policy")
def refund_policy_prompt(customer_name: str) -> list[base.Message]:
    """Generates a prompt with refund guidelines for a specific customer."""
    return [
        base.UserMessage(
            f"I need help with a refund for customer {customer_name}. "
            f"What is the refund policy?"
        ),
        base.AssistantMessage(
            f"I'll help you with the refund for {customer_name}. "
            f"POLICY: Refunds are only allowed for PENDING orders. "
            f"Shipped orders must be returned first before processing a refund."
        )
    ]

if __name__ == "__main__":
    mcp.run()
```

</p>
</details>
</details>

In [9]:
%%writefile my_server.py

## YOUR CODE HERE
from mcp.server.fastmcp import FastMCP
from mcp.server.fastmcp.prompts import base

# FastMCP is the high-level, beginner-friendly way to create MCP servers
# Think of it like Flask/FastAPI but for MCP protocol
mcp = FastMCP("Hello Server")

# ---- TOOLS (ACTIONS) -----

@mcp.tool(name="echo_tool")
def get_order_status(order_id: str) -> dict:
    """
    Retrieves the current status of an order.
    
    Args:
        order_id: The unique identifier of the order (e.g., ORD-123).
    """
    mock_db = {
        "ORD-123": "SHIPPED",
        "ORD-456": "PENDING",
        "ORD-999": "CANCELLED"
    }
    
    status = mock_db.get(order_id, "NOT_FOUND")
    return {"order_id": order_id, "status": status}


# --- RESOURCES (Data access) ---

@mcp.resource("system://logs")
def get_system_logs() -> str:
    """Read the latest system logs."""
    return "System logs: No critical errors today. All services operational."

# --- PROMPT (COntext templates) -----

@mcp.prompt("refund_policy")
def refund_policy_prompt(customer_name: str) -> list[base.Message]:
    """Generates a prompt with refund guidelines for a specific customer."""
    return [
        base.UserMessage(
            f"I need help with a refund for customer {customer_name}. "
            f"What is the refund policy?"
        ),
        base.AssistantMessage(
            f"I'll help you with the refund for {customer_name}. "
            f"POLICY: Refunds are only allowed for PENDING orders. "
            f"Shipped orders must be returned first before processing a refund."
        )
    ]



if __name__ == "__main__":
    mcp.run()

Overwriting my_server.py


## 🧠 Zadanie 3.1.2: Klient MCP

Teraz napisz klienta `my_client.py`, który połączy się z Twoim serwerem `server.py` (tym od systemu zamówień).

### Wymagania:
1.  Skonfiguruj `StdioServerParameters` tak, aby uruchamić `my_server.py`.
2.  W bloku sesji wykonaj następujące kroki (symulacja pracy agenta):
    *   **Discovery:** Wypisz listę dostępnych narzędzi.
    *   **Action:** Sprawdź status zamówienia "ORD-123" (użyj `call_tool`).
    *   **Knowledge:** Pobierz logi systemowe (użyj `read_resource`).
    *   **Context:** Pobierz prompt `refund_policy` dla klienta "Alice" (użyj `get_prompt`).

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈
    </summary>
<p>

- Upewnij się, że w `args` podajesz poprawną nazwę pliku: `["my_server.py"]`.
- Pamiętaj, że wszystkie metody sesji są asynchroniczne (`await`).
- Wynik `call_tool` znajduje się w polu `.content`, który jest listą (zazwyczaj interesuje nas element `[0].text`).

</p>
<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć rozwiązanie 👈
    </summary>
<p>


#### Odpowiedź:    
Upewnij się, że ta nazwa pliku (`my_server.py`) jest zgodna z nazwą pliku serwera utworzonego w poprzednim kroku.
    
    
```python
%%writefile my_client.py
import asyncio
from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession

async def run_agent_simulation():
    # Configure the client to run the python script as a subprocess
    server_params = StdioServerParameters(
        command="python",
        args=["my_server.py"],
        env=None
    )

    print("🔌 Connecting to Order System MCP...")
    
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # 1. Discovery - What can I do?
            tools = await session.list_tools()
            print(f"\n🛠️  Available Tools: {[t.name for t in tools.tools]}")
            
            # 2. Action - Check Order Status
            print("\n🤖 Agent: Checking status for ORD-123...")
            result = await session.call_tool("get_order_status", arguments={"order_id": "ORD-123"})
            print(f"📦 Tool Output: {result.content[0].text}")
            
            # 3. Knowledge - Read Logs
            print("\n🤖 Agent: Reading system logs...")
            logs = await session.read_resource("system://logs")
            print(f"📄 Resource Content: {logs.contents[0].text}")
            
            # 4. Context - Load Prompt
            print("\n🤖 Agent: Loading refund policy for Alice...")
            prompt = await session.get_prompt("refund_policy", arguments={"customer_name": "Alice"})
            
            # Print all messages in the prompt
            print("🧠 Prompt Messages:")
            for i, msg in enumerate(prompt.messages):
                role = msg.role  # "user" or "assistant"
                text = msg.content.text
                print(f"   [{role.upper()}]: {text[:80]}...")

if __name__ == "__main__":
    asyncio.run(run_agent_simulation())
```

</p>
</details>
</details>

In [7]:
%%writefile my_client.py
import asyncio
from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession

async def run_agent_simulation():
    # Configure the client to run the python script as a subprocess
    server_params = StdioServerParameters(
        command="python",
        args=["my_server.py"],
        env=None
    )

    print("🔌 Connecting to Order System MCP...")
    
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # 1. Discovery - What can I do?
            tools = await session.list_tools()
            print(f"\n🛠️  Available Tools: {[t.name for t in tools.tools]}")
            
            # 2. Action - Check Order Status
            print("\n🤖 Agent: Checking status for ORD-123...")
            result = await session.call_tool("get_order_status", arguments={"order_id": "ORD-123"})
            print(f"📦 Tool Output: {result.content[0].text}")
            
            # 3. Knowledge - Read Logs
            print("\n🤖 Agent: Reading system logs...")
            logs = await session.read_resource("system://logs")
            print(f"📄 Resource Content: {logs.contents[0].text}")
            
            # 4. Context - Load Prompt
            print("\n🤖 Agent: Loading refund policy for Alice...")
            prompt = await session.get_prompt("refund_policy", arguments={"customer_name": "Alice"})
            
            # Print all messages in the prompt
            print("🧠 Prompt Messages:")
            for i, msg in enumerate(prompt.messages):
                role = msg.role  # "user" or "assistant"
                text = msg.content.text
                print(f"   [{role.upper()}]: {text[:80]}...")

if __name__ == "__main__":
    asyncio.run(run_agent_simulation())

Overwriting my_client.py


Gratulacje! Właśnie zbudowałeś kompletny system: serwer MCP, który **implementuje logikę biznesową** zgodnie ze standardem MCP, oraz klienta MCP, który potrafi te możliwości konsumować. To nie jest teoretyczny model – stworzyłeś dwa niezależne byty, które potrafią nawiązać dialog, wymienić się informacjami o swoich zdolnościach (handshake) i wykonać konkretne zadania.

Teraz nadszedł czas na **weryfikację**. Musimy mieć pewność, że pliki `my_server.py` i `my_client.py` są poprawnie zapisane na dysku. Jeśli napotkałeś trudności lub Twój kod wymaga poprawki – to naturalny element procesu inżynierskiego. Śmiało skorzystaj z podpowiedzi lub po prostu skopiuj zweryfikowane rozwiązanie z sekcji "Odpowiedź". Kluczowe jest to, abyś w tym momencie dysponował poprawnym kodem, ponieważ za chwilę uruchomimy tę maszynę i zobaczymy protokół MCP w akcji.

Odpalamy!


In [10]:
!python my_client.py

🔌 Connecting to Order System MCP...
[12/25/25 18:01:38] INFO     Processing request of type            ]8;id=391244;file:///opt/conda/envs/python310/lib/python3.10/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=599330;file:///opt/conda/envs/python310/lib/python3.10/site-packages/mcp/server/lowlevel/server.py#674\674]8;;\
                             ListToolsRequest                                   

🛠️  Available Tools: ['echo_tool']

🤖 Agent: Checking status for ORD-123...
                    INFO     Processing request of type            ]8;id=696023;file:///opt/conda/envs/python310/lib/python3.10/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=973297;file:///opt/conda/envs/python310/lib/python3.10/site-packages/mcp/server/lowlevel/server.py#674\674]8;;\
                             CallToolRequest                                    
                    WARNING  Tool 'get_order_status' not listed,   ]8;id=508964;file:///opt/conda/

## Podsumowanie

W tej lekcji przeszliśmy drogę od teoretycznych rozważań o "agentach z rękami" do działającego kodu, który łączy dwa niezależne procesy. To był fundament, bez którego nie da się budować systemów Software 3.0.

Oto kluczowe wnioski, które warto zabrać ze sobą:

1. MCP to nie religia, tylko ustandaryzowany kabel USB-C dla AI, który pozwala nam przestać pisać setki niestandardowych integracji
2. Architektura MCP składa się zawsze z Hosta (centrala), Klienta (tłumacz) i Serwera (wykonawca), a każdy z nich ma swoją unikalną rolę w systemie
3. Narzędzia służą do działania, Zasoby do czytania, a Prompty do standaryzacji komunikacji i mylenie tych pojęć to prosty sposób na błędy logiczne
4. Transport oparty na stdio jest idealny do developmentu lokalnego, ale produkcyjne systemy muszą opierać się na stabilnym HTTP
5. Twoja pierwsza implementacja udowodniła, że agent może bezpiecznie korzystać z zewnętrznych funkcji bez znajomości ich wewnętrznego kodu

## Co dalej?

Mamy teraz działającego klienta i jeden serwer. Naturalnym odruchem inżyniera jest teraz chęć skalowania – podłączenia dziesięciu, pięćdziesięciu, a może stu serwerów MCP, aby nasz agent stał się "wszechmogący".

Brutalna prawda jest taka, że to najszybsza droga do katastrofy.

W kolejnej lekcji zmierzymy się z problemem "cyfrowego śmietnika". Zobaczysz, co dzieje się z modelem, gdy zalejemy go tysiącami definicji narzędzi z repozytoriów społeczności. Większość z nich jest niskiej jakości, a ich nadmiar powoduje, że agent traci zdolność logicznego rozumowania.

Nie chcemy budować systemów, które dławią się własnymi możliwościami. Dlatego w następnym kroku nauczysz się, jak budować inteligentne routery i bramy (Gateway), które filtrują szum i dostarczają agentowi tylko te narzędzia, których w danej sekundzie faktycznie potrzebuje. Zrozumiesz przyczyny, dla których "mniej znaczy więcej" i jak robić to profesjonalnie.

## Przydatne linki

1. [Introducing the Model Context Protocol](https://www.anthropic.com/news/model-context-protocol)
2. [One Year of MCP: November 2025 Spec Release](https://blog.modelcontextprotocol.io/posts/2025-11-25-first-mcp-anniversary/)
3. [Why MCP Deprecated SSE and Went with Streamable HTTP](https://blog.fka.dev/blog/2025-06-06-why-mcp-deprecated-sse-and-go-with-streamable-http/)
4. [A Critical Look at MCP](https://raz.sh/blog/2025-05-02_a_critical_look_at_mcp)
5. [Create 3D with Claude AI with Blender MCP - Full 26-min Tutorial](https://www.youtube.com/watch?v=lCyQ717DuzQ) + [github](https://github.com/ahujasid/blender-mcp)
6. [MCP Is Not Good Yet — David Cramer, Sentry](https://www.youtube.com/watch?v=FCi4jT86gSw)
7. [Model Context Protocol Blog](https://blog.modelcontextprotocol.io)
